# RG3 F1: 보정 순위3의 정상/가스 밀도비 앙상블

104번의3개조건부순위와 동일 CORAL/ref별가스방향 자체확장. 지도밀도비: 학습 정상과 가스 각각Gaussian KDE, 폭.3/.7/1.2 사전고정, log가스밀도-log정상밀도 sigmoid 후3개폭 평균. 실제 class prior를0.5로 놓는 점수이며 실제 가스확률이라고 주장하지 않음. 각 밀도는 학습ref RG3와 가중.25 보조CN7만 사용. 원래rank75 RF/ET 앙상블도 비교·조합.

평가RG3 RH620(정상601/가스19)/같은7개구간. CN7가스10은 보조학습만. 정상척도·정렬·밀도·가중치·임계값에서 외부 구간 제외. 내부OOF F1로 전문가 단독/쌍평균 및 임계값 선택. 같은 개발자료 중첩검증이며 새독립검증 아님. 공분산정렬원문: https://ojs.aaai.org/index.php/AAAI/article/view/10306


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import RobustScaler
from scipy.special import expit
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
a=np.load(t/'rg3_CN7_support_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19 and np.array_equal(a['labels'],y) and np.array_equal(a['timestamps'],times)
cn_y=a['CN7_auxiliary_labels'];cn=a['CN7_auxiliary_rank'][:,[6,8,9]]
assert len(cn)==705 and cn_y.sum()==10
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
def power(C,p):
 val,V=np.linalg.eigh(C);assert np.min(val)>0
 return (V*(val**p))@V.T
source_mean=cn[cn_y==0].mean(0)
Cs=LedoitWolf().fit(cn[cn_y==0]).covariance_+.01*np.eye(3)
source_whitener=power(Cs,-.5)
memo={};audit=[];align_audit=[]
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  n=sources[key];F=z['features_'+n+'_1'][:,[6,8,9]]
  normal=ref[y[ref]==0];positive=ref[y[ref]==1]
  mean=F[normal].mean(0)
  Ct=LedoitWolf().fit(F[normal]).covariance_+.01*np.eye(3)
  target_color=power(Ct,.5);target_white=power(Ct,-.5)
  src_direction=(cn[cn_y==1].mean(0)-source_mean)@source_whitener
  target_direction=(F[positive].mean(0)-mean)@target_white
  H=np.eye(3)
  if np.linalg.norm(src_direction)>1e-8 and np.linalg.norm(target_direction)>1e-8:
   u=src_direction/np.linalg.norm(src_direction);v=target_direction/np.linalg.norm(target_direction);w=u-v
   if np.linalg.norm(w)>1e-8:H=np.eye(3)-2*np.outer(w,w)/np.dot(w,w)
  values=[z['scores_'+n][:,1]]
  experiments=[('RG3_noise3',None,None)]
  for label,R in [('normal_CORAL',np.eye(3)),('class_direction_extension',H)]:
   A=source_whitener@R@target_color
   error=float(np.linalg.norm(A.T@Cs@A-Ct))
   assert error<1e-7 and np.linalg.norm(R.T@R-np.eye(3))<1e-7
   transformed=(cn-source_mean)@A+mean
   for only_gas in [False,True]:experiments.append((label,transformed,only_gas))
   align_audit.append(dict(train=len(ref),method=label,regularized_covariance_error=error,source_class_direction_norm=float(np.linalg.norm(src_direction)),target_class_direction_norm=float(np.linalg.norm(target_direction)),train_blocks=','.join(map(str,np.unique(groups[ref])))))
  for label,transformed,only_gas in experiments:
   if transformed is None:tr=F[ref];yy=y[ref];sw=np.ones(len(ref))
   else:
    ids=np.flatnonzero(cn_y==1) if only_gas else np.arange(len(cn))
    tr=np.r_[F[ref],transformed[ids]];yy=np.r_[y[ref],cn_y[ids]];sw=np.r_[np.ones(len(ref)),np.full(len(ids),.25)]
   preds=[]
   scaler=RobustScaler().fit(F[normal])
   tx=scaler.transform(tr);qx=scaler.transform(F)
   for bandwidth in [.3,.7,1.2]:
    positive_density=KernelDensity(bandwidth=bandwidth).fit(tx[yy==1],sample_weight=sw[yy==1])
    normal_density=KernelDensity(bandwidth=bandwidth).fit(tx[yy==0],sample_weight=sw[yy==0])
    contrast=positive_density.score_samples(qx)-normal_density.score_samples(qx)
    preds.append(expit(np.clip(contrast,-30,30)))
   values.append(np.mean(preds,axis=0))
  memo[key]=np.column_stack(values)
  audit.append(dict(RG3_train=len(ref),RG3_train_gas=int(y[ref].sum()),train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
names=['rank75_reference','RG3_noise3','CORAL_all_CN7','CORAL_CN7_gas','class_direction_all_CN7','class_direction_CN7_gas']
mixes=[np.eye(6)[i] for i in range(6)]+[(np.eye(6)[i]+np.eye(6)[j])/2 for i in range(6) for j in range(i+1,6)]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b);out=predict(ref,q);inner=np.full((len(ref),6),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in mixes:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda v:v[:3])
 named=[('inner_F1_selected_CORAL_density',best[5],best[4])]
 for i,name in enumerate(names):
  cut,_=cutfit(y[ref],inner[:,i]);named.append((name+'_inner_F1_cut',out[:,i],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(b),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
 (t/'rg3_CORAL_density_progress.json').write_text(str(int(b)),encoding='utf-8')
 print('Finished training-only CORAL block',b,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit'),(pd.DataFrame(align_audit),'alignment_audit')]:obj.to_csv(t/f'rg3_CORAL_density_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


cache={'labels':y,'groups':groups,'timestamps':times}
for j,(ref,score) in enumerate(memo.items()):cache[f'ref_{j}']=np.array(ref,int);cache[f'scores_{j}']=score
np.savez_compressed(t/'rg3_CORAL_density_fold_cache.npz',**cache)


Finished training-only CORAL block 1


Finished training-only CORAL block 2


Finished training-only CORAL block 3


Finished training-only CORAL block 4


Finished training-only CORAL block 5


Finished training-only CORAL block 6


Finished training-only CORAL block 7


                               model       AP  TP  FP  FN  TN       F1   recall
          CORAL_CN7_gas_inner_F1_cut 0.123180   5  32  14 569 0.178571 0.263158
          CORAL_all_CN7_inner_F1_cut 0.114471   5  34  14 567 0.172414 0.263158
             RG3_noise3_inner_F1_cut 0.132989   6  40  13 561 0.184615 0.315789
class_direction_CN7_gas_inner_F1_cut 0.105533   6  35  13 566 0.200000 0.315789
class_direction_all_CN7_inner_F1_cut 0.108793   5  31  14 570 0.181818 0.263158
     inner_F1_selected_CORAL_density 0.109918   5  36  14 565 0.166667 0.263158
       rank75_reference_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
